In [1]:
import pandas as pd

In [3]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

In [4]:
adnimerge.dtypes

RID                         int64
COLPROT                    object
VISCODE                    object
EXAMDATE           datetime64[ns]
AGE                       float64
EDUCATION                   int64
APOE4                     float64
CDRSB                     float64
ADAS11                    float64
ADAS13                    float64
MMSE                      float64
RAVLT_immediate           float64
FAQ                       float64
FSVERSION                 float64
IMAGEUID                  float64
Ventricles                float64
Hippocampus               float64
Entorhinal                float64
Fusiform                  float64
MidTemp                   float64
ICV                       float64
update_stamp       datetime64[ns]
GENDER_0                    int64
GENDER_1                    int64
MARRY_0                     int64
MARRY_1                     int64
MARRY_2                     int64
MARRY_3                     int64
ETHNICITY_0                 int64
ETHNICITY_1   

In [5]:
ptdemog.dtypes

PTID                    object
RID                      int64
VISCODE                 object
EXAMDATE        datetime64[ns]
PTDOB           datetime64[ns]
EDUCATION              float64
AGE                    float64
PTADBEG                float64
PTCOGBEG               float64
DX                     float64
HAS_QC_ERROR           float64
update_stamp    datetime64[ns]
GENDER_0                 int64
GENDER_1                 int64
MARRY_0                  int64
MARRY_1                  int64
MARRY_2                  int64
MARRY_3                  int64
ETHNICITY_0              int64
ETHNICITY_1              int64
RACE_0                   int64
RACE_1                   int64
RACE_2                   int64
RACE_3                   int64
RACE_4                   int64
RACE_5                   int64
VISIT_MONTH            float64
dtype: object

In [ ]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

In [40]:
KEYS = ["RID", "EXAMDATE"]

# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=KEYS,
    how="left",
    indicator=True
)

In [41]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
merged = merged.drop(columns="_merge")

_merge
left_only     8861
both           445
right_only       0
Name: count, dtype: int64


In [42]:
# --- 4. Individuazione righe duplicate sulla chiave, nel risultato finale ---
dup_mask = merged.duplicated(subset=KEYS, keep=False)   # keep=False: marca TUTTE le occorrenze coinvolte
dup_rows = merged[dup_mask].sort_values(KEYS)

In [43]:
print(f"Righe coinvolte in duplicati su {KEYS}: {dup_mask.sum()} su {len(merged)}")

Righe coinvolte in duplicati su ['RID', 'EXAMDATE']: 0 su 9306


In [44]:
# --- 5. Esportazione in CSV ---
merged.to_csv("ADNIMERGE_PTDEMOG_merged_01.csv", index=False, encoding="utf-8-sig")
dup_rows.to_csv("ADNIMERGE_PTDEMOG_duplicati_01.csv", index=False, encoding="utf-8-sig")

In [45]:
print(f"\nFile salvato: ADNIMERGE_PTDEMOG_merged.csv — {merged.shape[0]} righe, {merged.shape[1]} colonne")
print(f"File duplicati salvato: ADNIMERGE_PTDEMOG_duplicati.csv — {dup_rows.shape[0]} righe")


File salvato: ADNIMERGE_PTDEMOG_merged.csv — 9306 righe, 66 colonne
File duplicati salvato: ADNIMERGE_PTDEMOG_duplicati.csv — 0 righe


Controlla PTDOB

In [30]:
import pandas as pd

df = pd.read_csv("ADNIMERGE_cleaned_02.csv")  # o il nome/percorso corretto del tuo file PTDEMOG

In [31]:
# --- Ispezione colonna EXAMDATE prima del parsing ---
print("Dtype pandas:", df["EXAMDATE"].dtype)
print("\nPrimi valori non nulli:")
print(df["EXAMDATE"].dropna().unique()[:10])

print("\nType dell'oggetto Python contenuto (primo valore valido):")
primo_valido = df["EXAMDATE"].dropna().iloc[0]
print(type(primo_valido), "->", primo_valido)

print("\nConteggio valori nulli:", df["EXAMDATE"].isna().sum())

Dtype pandas: str

Primi valori non nulli:
<StringArray>
['2005-09-08', '2005-09-12', '2006-03-13', '2006-09-12', '2007-09-12',
 '2005-11-08', '2006-05-02', '2006-11-14', '2007-05-14', '2008-11-18']
Length: 10, dtype: str

Type dell'oggetto Python contenuto (primo valore valido):
<class 'str'> -> 2005-09-08

Conteggio valori nulli: 0


In [23]:
print(df["EXAMDATE"].dropna().unique()[:10])
print(df["update_stamp"].dropna().unique()[:10])

<StringArray>
['2005-09-08', '2005-09-12', '2006-03-13', '2006-09-12', '2007-09-12',
 '2005-11-08', '2006-05-02', '2006-11-14', '2007-05-14', '2008-11-18']
Length: 10, dtype: str
<StringArray>
['2023-07-07 04:59:40', '2023-07-07 04:59:41', '2023-07-09 05:25:23',
 '2023-08-18 05:00:20', '2023-07-07 04:59:42', '2023-07-07 04:59:43',
 '2023-07-07 04:59:44', '2023-07-09 05:25:25', '2023-07-07 04:59:45',
 '2023-07-07 04:59:46']
Length: 10, dtype: str
